# ML path prediction with the feature store

This notebook brings together two analyses that have so far been kept apart:

- **`Feature_Store_Datasets.ipynb`** builds a feature store from the raw capture
  (`banda.bwm`, `eventos.txt`, `rotas.txt`, `config.json`) of every dataset in
  `datasets/D*`. The store has latency and bandwidth per route and second, the
  sliding-window features (5 s, 15 s, 30 s), the cross features, and the target
  `melhor_rota`.
- **`Análise_de_modelos_de_ML_para_previsão_de_caminhos.ipynb`** defines how the
  models are evaluated: seven classifiers and, in its predictive section, the
  protocol of `protocolo_temporal.py`. Under that protocol the target is shifted
  to `t + H`, train and test are split chronologically per dataset with an
  `H`-second gap between them, cross-validation uses `TimeSeriesSplit`, every
  model is preceded by a `StandardScaler`, and the persistence and majority-class
  baselines are reported.

Here the feature store supplies the input and the predictive protocol does the
evaluation. Neither source notebook is changed: this notebook imports the same
modules and writes its outputs to a folder of its own.

| part | taken from | what changes here |
|---|---|---|
| store construction and loading | `Feature_Store_Datasets.ipynb`, §1–5 | the store is built only when it is missing (`REBUILD_STORE`) |
| wide layout (one row per instant, one column per feature × route) | `pipeline.construir_wide` | used for every feature set, not only for the bandwidth candidates |
| 7 classifiers + `StandardScaler` | ML notebook, *Pipeline com protocolo preditivo* | `SVC` without `probability=True`, because only `predict` is used |
| target shift, chronological split, baselines | `protocolo_temporal.py` | unchanged |
| `TimeSeriesSplit` cross-validation, confusion grid | ML notebook | unchanged |
| per-scenario grouping (D*x* + D*x*b) | `Feature_Store_Datasets.ipynb`, §8 | used for every result |
| permutation importance in the predictive regime | `Feature_Store_Datasets.ipynb`, §9 | uses all 7 models' protocol, reports the std across repeats |
| cross-dataset generalisation (paper's Cenário 2) | ML notebook, *Reprodução da Tabela 2* | uses store features, no clock columns, and optionally `H > 0` |

**Left out on purpose:**
- The contemporaneous experiments with `shuffle=True` (`train_test_split`,
  `StratifiedKFold`). Both source notebooks show that shuffling inflates
  accuracy, and they keep those experiments for comparison with the paper.
- The `temporal_features` (hour, minute, second, ...). Under a chronological
  split they identify the test block, so the model learns the clock instead of
  the network.
- The mice/elephant multi-objective sections. They depend on the per-flow label
  files (`rotulos_por_fluxo*.txt`), which the store does not produce.

**Target.** `melhor_rota` comes from `pipeline.derivar_alvo`: it is the
`argmin(latencia_ms)` of the instant, without the tie-break used by
`rotulos_menor_latencia_historico.txt`. At `H = 0` the target is therefore a
deterministic function of the latency features, so the `H = 0` column is a
reference point, not a forecast.

**D3a and D4a.** These two datasets are routes `h11_h61` and `h12_h62` of D3 and
D4 and have no raw capture of their own. Their stores are built by
`feature_store/build_route_subset.py` from the raw capture of D3 and D4, with
every store column, and the target is recomputed over the two kept routes
(section 2.1). Each of them is its own scope and is never pooled with D3 or D4,
since that would count the same instants twice.

## Contents
1. Setup
2. Feature store
3. Feature sets
4. Evaluation environment
5. Class distribution
6. Horizon sweep
7. Feature-set comparison at the reference horizon
8. Reference case in detail
9. Feature importance in the predictive regime
10. Cross-dataset generalisation
11. Sliding training window W
12. Notes and limitations

## 1. Setup

The project root is found from the directories that exist only there, so the
notebook runs from the root or from any subfolder.

`protocolo_temporal.py` now lives in `scripts/evaluation/`, but
`feature_store/pipeline.py` still imports it by its plain name. The cell below
therefore puts `scripts/evaluation/` on `sys.path` before importing the
pipeline.

In [ ]:
import os
import sys
from pathlib import Path


def project_root(markers=('prototipo', 'feature_store', 'datasets')):
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if all((candidate / marker).is_dir() for marker in markers):
            return candidate
    raise RuntimeError(
        f'project root not found from {Path.cwd()}: expected a folder holding {markers}'
    )


ROOT = project_root()
os.chdir(ROOT)

for folder in (ROOT / 'feature_store', ROOT / 'scripts' / 'evaluation'):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

print(f'project root : {ROOT}')
print(f'interpreter  : {sys.executable}')
print(f'python       : {sys.version.split()[0]}')

In [ ]:
import contextlib
import importlib
import io
import re
import warnings

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from sklearn import svm, tree
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import protocolo_temporal as pt
import pipeline as fs

importlib.reload(pt)
importlib.reload(fs)

warnings.filterwarnings('ignore', category=UserWarning)
# Wide frames with ~100 columns trigger it on every assign; harmless here
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 80)

print(f'pandas {pd.__version__} | numpy {np.__version__}')
print(f'feature store : {fs.__file__}')
print(f'protocol      : {pt.__file__}')

In [ ]:
# Scenarios: each pair holds the same kind of background traffic, collected on the
#   original (Dx) and on the modified (Dxb) link latencies. Same grouping as
#   Feature_Store_Datasets.ipynb.
SCENARIOS = {
    'D1 (baseline)': ['D1', 'D1b'],
    'D2 (constant background)': ['D2', 'D2b'],
    'D3 (long iperf flow)': ['D3', 'D3b'],
    'D4 (concurrent iperf flows)': ['D4', 'D4b'],
    # Route subsets of D3 and D4 (section 2.1). Kept apart from D3 and D4: they
    #   are routes 1 and 2 of those same collections.
    'D3a (2 routes)': ['D3a'],
    'D4a (2 routes)': ['D4a'],
}

# Adds a scope that pools every four-route dataset. Off by default: with ~29k
#   instants the SVC fits of the horizon sweep take far longer than the scenarios.
INCLUDE_POOLED_SCOPE = False
POOLED_SCOPE = 'all datasets'

# Prediction horizons (s). H = 0 is the contemporaneous task under a chronological
#   split, kept as the reference that isolates the effect of the shift.
HORIZONS = [0, 1, 5, 15, 30, 60]
H_REF = 30

TRAIN_FRACTION = 0.8
RANDOM_STATE = 1234
CV_FOLDS = 5

# Same floor as VAR_SMOOTHING_PRED in the ML notebook: with the default 1e-9 a
#   class with one or two training samples collapses into a density spike that
#   beats every prior (D1 with bandwidth predicted route 3 on 76% of the test).
VAR_SMOOTHING = 5e-2

MODEL_NAMES = ['LogisticRegression', 'KNeighbors', 'SVC', 'DecisionTree',
               'ExtraTrees', 'RandomForest', 'GaussianNB']

# Rebuild the store of every dataset even when feature_store/data/<D>/ exists
REBUILD_STORE = False
STORE_ROOT = Path('feature_store') / 'data'

# Every output of this notebook goes here, never into the folders of the others
RESULTS_DIR = ROOT / 'resultados_am' / 'CenarioFeatureStore'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Case examined in detail in section 8
REF_SCOPE = 'D4 (concurrent iperf flows)'
REF_FEATURE_SET = 'store: all features'

# Section 9
IMPORTANCE_FEATURE_SETS = ['store: bandwidth features', 'store: all features']
IMPORTANCE_REPEATS = 10

# Section 10: train on one dataset, test on whole datasets of other scenarios
CROSS_TRAIN = 'D1'
CROSS_TEST = {
    'original latencies': ['D2', 'D3', 'D4'],
    'modified latencies': ['D2b', 'D3b', 'D4b'],
}
CROSS_HORIZONS = [0, H_REF]

print(f'results folder: {RESULTS_DIR}')

## 2. Feature store

A dataset goes through `fs.construir` when it holds the four inputs that
function needs: `banda.bwm`, `eventos.txt`, `rotas.txt` and `config.json`. The
store of each such dataset is persisted in `feature_store/data/<D>/` by
`Feature_Store_Datasets.ipynb`. A store that already exists is read back as it
is, and `REBUILD_STORE = True` forces a rebuild from the raw capture.

D3a and D4a hold only the consolidated wide CSVs. They are built as route
subsets in section 2.1.

In [ ]:
REQUIRED_INPUTS = ['banda.bwm', 'eventos.txt', 'rotas.txt', 'config.json']


def has_raw_capture(folder):
    return all((folder / name).exists() for name in REQUIRED_INPUTS)


def store_is_built(dataset):
    return (STORE_ROOT / dataset / 'csv' / 'consolidado.csv').exists()


folders = sorted(p for p in Path('datasets').glob('D*') if p.is_dir())
DATASETS = [p.name for p in folders if has_raw_capture(p)]
print(f'raw capture ({len(DATASETS)}) : {DATASETS}')
print(f'no raw capture  : {[p.name for p in folders if not has_raw_capture(p)]} (see section 2.1)')

built, reused, failures = [], [], {}
for dataset in DATASETS:
    if store_is_built(dataset) and not REBUILD_STORE:
        reused.append(dataset)
        continue
    folder = f'datasets/{dataset}'
    try:
        fs.construir(
            pasta_relatorios=folder,
            caminho_config=f'{folder}/config.json',
            pasta_destino=str(STORE_ROOT / dataset),
            arquivo_banda=f'{folder}/banda.bwm',
        )
        built.append(dataset)
    except Exception as error:
        failures[dataset] = error

print(f'reused : {reused}')
print(f'built  : {built}')
for dataset, error in failures.items():
    print(f'FAILED : {dataset}: {error}')

DATASETS = [d for d in DATASETS if d not in failures]

In [ ]:
# Long layout: one row per (dataset, route, second), every store column
parts = []
for dataset in DATASETS:
    part = fs.get_features(pasta_origem=str(STORE_ROOT / dataset))
    part.insert(0, 'dataset_id', dataset)
    parts.append(part)

store_long = pd.concat(parts, ignore_index=True)

# Instants without a label (ping start-up) cannot be used as y
base = store_long.dropna(subset=['melhor_rota']).copy()
base['melhor_rota'] = base['melhor_rota'].astype(int)

summary = base.groupby('dataset_id').agg(
    routes=('rota_id', 'nunique'),
    instants=('ts_epoch', 'nunique'),
    rows=('ts_epoch', 'size'),
).reindex(DATASETS)

print(f'store: {store_long.shape[1]} columns, {len(store_long)} rows '
      f'({len(store_long) - len(base)} without a label)')
summary

### 2.1 Route subsets: D3a and D4a

`feature_store/build_route_subset.py` builds the store of any subset of the
paths of a collected dataset from that dataset's own raw capture. It runs
`fs.construir` on the original dataset, keeps the chosen routes, and recomputes
`melhor_rota` over them only. This is exact. Every other step works on one
route at a time: clock alignment (from `banda.bwm` and `eventos.txt` only),
bandwidth per interface and per route, latency, windows, deltas and cross
features. The only quantity that compares routes is the target. The kept
routes are numbered 1..k in sorted order of `rota_id`, and the mapping is
written to `subset.json` next to the store.

`ROUTE_SUBSETS` lists the subsets this notebook uses. Each one is persisted in
`feature_store/data/<name>/`, and a subset store that already exists is reused
unless `REBUILD_STORE = True`. The same script builds any other subset from the
command line, for example
`python feature_store/build_route_subset.py D4 2 4 --name D4_r24`. Adding the
result here and to `SCENARIOS` puts it through every section below.

When the script was written, the subsets were checked against the files that
already existed in `datasets/D3a` and `datasets/D4a`:
- latency matched value for value on every common instant;
- the target agreed 100% with `labels_h1_h6_12.txt` / `rotulos_h1_h6_12.txt`.

Bandwidth does not match those files and is not meant to. The old CSVs came
from the prototype's parser (Mibit/s against a decimal capacity, clipped at 0,
gaps interpolated), and the store reads `banda.bwm` in decimal Mbps.

In [ ]:
from build_route_subset import KNOWN_SUBSETS, build_route_subset

# name -> (source dataset, routes kept)
ROUTE_SUBSETS = dict(KNOWN_SUBSETS)

SUBSET_DATASETS = []
for name, (source, routes) in ROUTE_SUBSETS.items():
    if source not in DATASETS:
        print(f'{name}: source {source} has no store, skipped')
        continue
    if store_is_built(name) and not REBUILD_STORE:
        print(f'{name}: reused feature_store/data/{name} ({source}, routes {routes})')
    else:
        build_route_subset(source, routes, name=name, store_root=str(STORE_ROOT))
        print(f'{name}: built from {source}, routes {routes}')
    SUBSET_DATASETS.append(name)

subset_parts = []
for name in SUBSET_DATASETS:
    part = fs.get_features(pasta_origem=str(STORE_ROOT / name))
    part.insert(0, 'dataset_id', name)
    subset_parts.append(part)

if subset_parts:
    subset_long = pd.concat(subset_parts, ignore_index=True)
    subset_base = subset_long.dropna(subset=['melhor_rota']).copy()
    subset_base['melhor_rota'] = subset_base['melhor_rota'].astype(int)
    base = pd.concat([base, subset_base], ignore_index=True)

ALL_DATASETS = DATASETS + SUBSET_DATASETS
base.groupby('dataset_id').agg(
    routes=('rota_id', 'nunique'),
    instants=('ts_epoch', 'nunique'),
    rows=('ts_epoch', 'size'),
).reindex(ALL_DATASETS)

In [ ]:
# Structural checks, as in section 5 of Feature_Store_Datasets.ipynb
EXPECTED_ROUTES = {d: 4 for d in DATASETS}
EXPECTED_ROUTES.update({name: len(ROUTE_SUBSETS[name][1]) for name in SUBSET_DATASETS})

for dataset, group in base.groupby('dataset_id'):
    routes = EXPECTED_ROUTES[dataset]
    assert group.rota_id.nunique() == routes, f'{dataset}: expected {routes} routes'
    counts = group.rota_id.value_counts()
    assert counts.max() - counts.min() <= 10, \
        f'{dataset}: routes with very different row counts: {counts.to_dict()}'
    assert set(group.melhor_rota.unique()) <= set(range(1, routes + 1)), \
        f'{dataset}: unexpected label'

print(f'checks passed for {len(ALL_DATASETS)} datasets')

## 3. Feature sets

Every feature set becomes a wide frame built by `fs.construir_wide`: one row per
`(dataset_id, ts_epoch)` and one column `<feature>__<route>` per feature and
route, so that the model can compare the routes of the same instant. Instants
where any route lacks any feature are dropped. For that reason the sets do not
all have exactly the same rows, and every baseline below is measured on the
rows of its own set.

| set | columns per route | what it answers |
|---|---|---|
| `latency (instant)` | `latencia_ms` | the paper's input (minus the clock), now read from the store |
| `bandwidth (instant)` | `gargalo_Mbps` | the bottleneck alone, the analogue of `banda_rotas_h1_h6.csv` |
| `latency + bandwidth (instant)` | both | the combined source of the ML notebook (`METRICA_PRED = ('latencia', 'banda')`) |
| `store: bandwidth features` | every bandwidth column, instant and windowed | the candidates of `Feature_Store_Datasets.ipynb` (no `latencia_*`, no `bdp`) |
| `store: all features` | every store column | what the store adds on top of the instant values |

Datasets are pivoted in groups that share the same set of routes. A single
pivot would leave the instants of a two-route subset without routes 3 and 4,
and `construir_wide` would drop them all. Every group has every store column,
so the feature sets are the same for all of them. Only the number of columns
of `X` changes with the number of routes.

In [ ]:
NON_FEATURE = {'dataset_id', 'run_id', 'rota_id', 'ts_epoch', 'datetime_utc',
               'tempo_relativo_s', 'melhor_rota'}

store_columns = [c for c in base.columns if c not in NON_FEATURE and base[c].notna().any()]


def uses_latency(column):
    return column.startswith('latencia_') or column == 'bdp'


FEATURE_SETS = {
    'latency (instant)': ['latencia_ms'],
    'bandwidth (instant)': ['gargalo_Mbps'],
    'latency + bandwidth (instant)': ['latencia_ms', 'gargalo_Mbps'],
    'store: bandwidth features': [c for c in store_columns if not uses_latency(c)],
    'store: all features': store_columns,
}

# Datasets grouped by the routes they hold, e.g. ('h11_h61', 'h12_h62')
route_set_of = base.groupby('dataset_id').rota_id.agg(lambda s: tuple(sorted(s.unique())))
ROUTE_GROUPS = {
    ' + '.join(routes): base[base.dataset_id.isin(members.index)]
    for routes, members in route_set_of.groupby(route_set_of)
}

wide_frames, set_sizes = {}, []
for name, columns in FEATURE_SETS.items():
    pieces = []
    for group, frame in ROUTE_GROUPS.items():
        with contextlib.redirect_stdout(io.StringIO()) as log:
            piece = fs.construir_wide(frame, columns)
        if log.getvalue():
            print(f'{name} | {group}: {log.getvalue().strip()}')
        pieces.append(piece)
        set_sizes.append({'feature set': name, 'routes': group,
                          'datasets': ', '.join(sorted(piece.dataset_id.unique())),
                          'columns of X': len(pt.colunas_de_feature(piece)),
                          'instants': len(piece)})
    # Route columns a group does not have stay NaN on its rows; scope_frame drops them
    wide_frames[name] = pd.concat(pieces, ignore_index=True)

pd.DataFrame(set_sizes).set_index(['feature set', 'routes'])

In [ ]:
SCOPES = {name: [d for d in members if d in ALL_DATASETS] for name, members in SCENARIOS.items()}
SCOPES = {name: members for name, members in SCOPES.items() if members}
if INCLUDE_POOLED_SCOPE:
    # Four-route datasets only: mixing route counts leaves NaN columns in X
    SCOPES[POOLED_SCOPE] = list(DATASETS)

for scope, members in SCOPES.items():
    counts = {EXPECTED_ROUTES[d] for d in members}
    assert len(counts) == 1, f'{scope}: mixes datasets with {sorted(counts)} routes'


def scope_frame(feature_set, scope):
    frame = wide_frames[feature_set]
    frame = frame[frame.dataset_id.isin(SCOPES[scope])]
    # Drops the columns of routes or features this scope does not have
    return frame.dropna(axis=1, how='all').reset_index(drop=True)


def slug(text):
    return re.sub(r'[^A-Za-z0-9]+', '_', text).strip('_')


def metric_of(column):
    # 'gargalo_min_curta_15s__h11_h61' -> 'gargalo_min_curta_15s'
    return column.split('__')[0]


print({scope: members for scope, members in SCOPES.items()})

## 4. Evaluation environment

These are the same seven classifiers as the predictive section of the ML
notebook, each preceded by a `StandardScaler`. `SVC` drops
`probability=True`: `predict` does not use the probability calibration, so the
predictions stay the same and the fit skips its internal five-fold run.

`evaluate_predictive` applies `pt.preparar_treino_teste`, which shifts the
target to `t + H` and splits each dataset chronologically with an `H`-second
gap. It then trains every model on the training block and measures it on the
test block, alongside two baselines:

- **persistence**: the best route in `H` seconds is the best route now. This is
  what a reactive controller gets without any model.
- **majority class**: the most frequent class of the **training** block.
  Choosing it from the test block would already be leakage.

A training block with a single class (possible in D1) makes some models fail.
Those models get `NaN` instead of stopping the run.

In [ ]:
def make_pipelines(names=MODEL_NAMES, random_state=RANDOM_STATE, n_jobs=-1, n_samples=None):
    # n_samples: size of the training set, when known to be small (section 11).
    #   KNN cannot ask for more neighbours than there are samples.
    neighbours = 5 if n_samples is None else max(1, min(5, n_samples))
    catalogue = {
        'LogisticRegression': lambda: LogisticRegression(max_iter=1000),
        'KNeighbors': lambda: KNeighborsClassifier(n_neighbors=neighbours),
        'SVC': lambda: svm.SVC(random_state=random_state),
        'DecisionTree': lambda: tree.DecisionTreeClassifier(random_state=random_state),
        'ExtraTrees': lambda: ExtraTreesClassifier(random_state=random_state, n_jobs=n_jobs),
        'RandomForest': lambda: RandomForestClassifier(random_state=random_state, n_jobs=n_jobs),
        'GaussianNB': lambda: GaussianNB(var_smoothing=VAR_SMOOTHING),
    }
    return {
        name: Pipeline([('scaler', StandardScaler()), ('classifier', catalogue[name]())])
        for name in names
    }


def split_blocks(wide, horizon_s, fraction=TRAIN_FRACTION, quiet=True):
    # deslocar_alvo prints how many rows it drops; silenced inside the sweeps
    sink = io.StringIO() if quiet else sys.stdout
    with contextlib.redirect_stdout(sink):
        frame, train, test = pt.preparar_treino_teste(wide, horizon_s, fracao_treino=fraction)

    features = pt.colunas_de_feature(frame)
    X = frame[features].to_numpy(dtype=float)
    y = frame['melhor_rota'].to_numpy(dtype=int)
    current = frame[pt.COLUNA_ALVO_ATUAL].to_numpy(dtype=int)
    train, test = train.to_numpy(), test.to_numpy()
    return {
        'features': features,
        'X_train': X[train], 'y_train': y[train],
        'X_test': X[test], 'y_test': y[test],
        'current_test': current[test],
    }


def evaluate_predictive(wide, horizon_s, model_names=MODEL_NAMES, keep_models=False, quiet=True):
    blocks = split_blocks(wide, horizon_s, quiet=quiet)
    X_train, y_train = blocks['X_train'], blocks['y_train']
    X_test, y_test = blocks['X_test'], blocks['y_test']

    rows, trained = [], {}
    for name, model in make_pipelines(model_names).items():
        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
            if keep_models:
                trained[name] = model
        except ValueError as error:
            print(f'  {name}: not trained ({error})')
            accuracy = f1 = np.nan
        rows.append({'model': name, 'accuracy': accuracy, 'f1_macro': f1})

    majority_class = int(pd.Series(y_train).value_counts().idxmax())
    return {
        **blocks,
        'horizon_s': horizon_s,
        'results': pd.DataFrame(rows),
        'persistence': float((blocks['current_test'] == y_test).mean()),
        'majority': float((y_test == majority_class).mean()),
        'majority_class': majority_class,
        'n_train': len(y_train),
        'n_test': len(y_test),
        'classes': sorted(set(y_train.tolist()) | set(y_test.tolist())),
        'trained': trained,
    }


print('evaluation helpers defined')

In [ ]:
def temporal_cross_validation(X_train, y_train, pipelines, folds=CV_FOLDS):
    # Each fold trains on a prefix of the training block and validates on the
    #   stretch right after it; StratifiedKFold would shuffle the instants and bring
    #   back, inside the training block, the leakage the chronological split removed.
    splitter = TimeSeriesSplit(n_splits=folds)
    rows = []
    for name, pipeline in pipelines.items():
        scores = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
        try:
            for fit_idx, val_idx in splitter.split(X_train):
                model = clone(pipeline).fit(X_train[fit_idx], y_train[fit_idx])
                y_pred, y_true = model.predict(X_train[val_idx]), y_train[val_idx]
                scores['accuracy'].append(accuracy_score(y_true, y_pred))
                scores['precision'].append(precision_score(y_true, y_pred, average='macro', zero_division=0))
                scores['recall'].append(recall_score(y_true, y_pred, average='macro', zero_division=0))
                scores['f1'].append(f1_score(y_true, y_pred, average='macro', zero_division=0))
            rows.append({
                'model': name,
                'cv_accuracy_mean': np.mean(scores['accuracy']),
                'cv_accuracy_std': np.std(scores['accuracy']),
                'cv_precision_macro': np.mean(scores['precision']),
                'cv_recall_macro': np.mean(scores['recall']),
                'cv_f1_macro': np.mean(scores['f1']),
            })
        except ValueError as error:
            print(f'{name}: not evaluated ({error})')
            rows.append({'model': name})
    return pd.DataFrame(rows).set_index('model')


def plot_confusion_grid(trained, X, y, labels, title='', columns=4, save_path=None):
    # Rows are the actual routes and columns the predicted ones, as in plot_matrix
    matrices = {name: confusion_matrix(y, model.predict(X), labels=labels)
                for name, model in trained.items()}
    rows = int(np.ceil(len(matrices) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(3.2 * columns, 3.4 * rows), squeeze=False)
    ticks = [str(label) for label in labels]

    for axis, (name, matrix) in zip(axes.ravel(), matrices.items()):
        axis.matshow(matrix, cmap=plt.cm.YlGnBu, alpha=0.8)
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                axis.text(j, i, matrix[i, j], ha='center', va='center', fontsize=7)
        axis.set_xticks(range(len(ticks)), ticks, fontsize=7)
        axis.set_yticks(range(len(ticks)), ticks, fontsize=7)
        axis.xaxis.set_ticks_position('bottom')
        total = matrix.sum()
        axis.set_title(f'{name}\nacc {matrix.diagonal().sum() / total:.4f}' if total else name, fontsize=9)
        axis.set_xlabel('predicted', fontsize=8)
        axis.set_ylabel('actual', fontsize=8)

    for axis in axes.ravel()[len(matrices):]:
        axis.axis('off')
    if title:
        fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    return matrices


print('cross-validation and plotting helpers defined')

## 5. Class distribution

This shows the share of each route in the full series, in the training block
and in the test block, at the reference horizon and on the rows of
`store: all features`, the most restrictive set. Read it before any accuracy.
When one route wins almost every instant (D1), a high accuracy only restates
the imbalance. When the dominant route changes between the two blocks, the
majority baseline stops measuring imbalance and starts measuring the regime
shift.

In [ ]:
def class_distribution(wide, horizon_s=H_REF, fraction=TRAIN_FRACTION):
    with contextlib.redirect_stdout(io.StringIO()):
        frame, train, test = pt.preparar_treino_teste(wide, horizon_s, fracao_treino=fraction)
    target = frame['melhor_rota'].astype(int)
    blocks = {
        'full %': target,
        'train %': target[train.to_numpy()],
        'test %': target[test.to_numpy()],
    }
    table = pd.DataFrame({
        name: block.value_counts(normalize=True).mul(100).round(2) for name, block in blocks.items()
    }).fillna(0.0).sort_index()
    table.index = [f'route {route}' for route in table.index]
    table.loc['instants'] = [len(block) for block in blocks.values()]
    return table


distribution = pd.concat(
    {scope: class_distribution(scope_frame('store: all features', scope)) for scope in SCOPES},
    names=['scope', ''],
)
distribution.to_csv(RESULTS_DIR / f'class_distribution_H{H_REF}.csv')
distribution

## 6. Horizon sweep

The sweep runs every scope × feature set × horizon × model. The figure that
matters is not the accuracy itself but the **gain over persistence**. A model
that does not beat "the best route in `H` seconds is the best route now" is not
predicting anything a reactive controller would not get for free.

With the default settings this is 6 scopes × 5 sets × 6 horizons × 7 models =
1260 fits, and it takes tens of minutes. The two-route scopes are the
cheapest, since they have half the instants of a scenario and half the columns. The results are saved to
`sweep_accuracy.csv` and `sweep_baselines.csv`.

In [ ]:
sweep_rows, baseline_rows = [], []

for scope in SCOPES:
    for feature_set in FEATURE_SETS:
        wide = scope_frame(feature_set, scope)
        for H in HORIZONS:
            r = evaluate_predictive(wide, H)
            keys = {'scope': scope, 'feature_set': feature_set, 'horizon_s': H}
            sweep_rows += [{**keys, **row} for row in r['results'].to_dict('records')]
            baseline_rows.append({**keys, 'persistence': r['persistence'], 'majority': r['majority'],
                                  'n_train': r['n_train'], 'n_test': r['n_test']})
        print(f'{scope:28s} | {feature_set:30s} | {len(HORIZONS)} horizons done')

sweep = pd.DataFrame(sweep_rows)
baselines = pd.DataFrame(baseline_rows)
sweep.to_csv(RESULTS_DIR / 'sweep_accuracy.csv', index=False)
baselines.to_csv(RESULTS_DIR / 'sweep_baselines.csv', index=False)
print(f'saved to {RESULTS_DIR}')

In [ ]:
# Best model of each (scope, feature set, horizon), next to the two baselines
KEYS = ['scope', 'feature_set', 'horizon_s']
valid = sweep.dropna(subset=['accuracy'])
best = (valid.loc[valid.groupby(KEYS)['accuracy'].idxmax()]
        .merge(baselines, on=KEYS)
        .rename(columns={'model': 'best_model'}))
best['gain_over_persistence'] = best['accuracy'] - best['persistence']
best.to_csv(RESULTS_DIR / 'sweep_best_model.csv', index=False)

ORDER = {'scope': list(SCOPES), 'feature_set': list(FEATURE_SETS)}


def ordered(table):
    return table.reindex(pd.MultiIndex.from_product([ORDER['scope'], ORDER['feature_set']],
                                                    names=['scope', 'feature_set']))


print('Gain of the best model over persistence (accuracy points), by horizon:')
ordered(best.pivot_table(index=['scope', 'feature_set'], columns='horizon_s',
                         values='gain_over_persistence')).round(4)

In [ ]:
print('Accuracy of the best model, by horizon (persistence in the last block):')
accuracy_table = ordered(best.pivot_table(index=['scope', 'feature_set'], columns='horizon_s', values='accuracy'))
persistence_table = (baselines[baselines.feature_set == 'store: all features']
                     .pivot_table(index='scope', columns='horizon_s', values='persistence')
                     .reindex(ORDER['scope']))
persistence_table.index = pd.MultiIndex.from_product([persistence_table.index, ['baseline: persistence']],
                                                     names=['scope', 'feature_set'])
pd.concat([accuracy_table, persistence_table]).sort_index(level=0, sort_remaining=False).round(4)

In [ ]:
fig, axes = plt.subplots(int(np.ceil(len(SCOPES) / 2)), 2, figsize=(14, 4.6 * np.ceil(len(SCOPES) / 2)),
                         squeeze=False, sharey=True)

for axis, scope in zip(axes.ravel(), SCOPES):
    for feature_set in FEATURE_SETS:
        rows = best[(best.scope == scope) & (best.feature_set == feature_set)].sort_values('horizon_s')
        axis.plot(rows.horizon_s, rows.accuracy, marker='o', label=feature_set)
    reference = baselines[(baselines.scope == scope) & (baselines.feature_set == 'store: all features')]
    reference = reference.sort_values('horizon_s')
    axis.plot(reference.horizon_s, reference.persistence, 'k--', marker='s', linewidth=2,
              label='baseline: persistence')
    axis.plot(reference.horizon_s, reference.majority, ':', color='gray', marker='^', linewidth=2,
              label='baseline: majority class')
    axis.set_title(scope, fontsize=11)
    axis.set_xlabel('prediction horizon H (s)')
    axis.set_ylabel('accuracy of the best model (test block)')
    axis.grid(alpha=0.3)

for axis in axes.ravel()[len(SCOPES):]:
    axis.axis('off')
axes.ravel()[0].legend(fontsize=8, loc='lower left')
fig.suptitle('Best model per feature set vs. horizon (target shifted to t+H, chronological split)', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sweep_best_model.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Feature-set comparison at the reference horizon

This section reads the sweep at `H_REF`, one model at a time. The question is
whether the windowed features of the store add anything on top of the instant
values, and whether that gain holds across models or comes from one of them
only.

In [ ]:
at_ref = sweep[sweep.horizon_s == H_REF]
per_model = (at_ref.pivot_table(index=['scope', 'model'], columns='feature_set', values='accuracy')
             .reindex(columns=list(FEATURE_SETS)))
per_model = per_model.reindex(pd.MultiIndex.from_product([ORDER['scope'], MODEL_NAMES],
                                                         names=['scope', 'model']))
per_model.to_csv(RESULTS_DIR / f'accuracy_per_model_H{H_REF}.csv')
print(f'Accuracy per model and feature set, H={H_REF}s:')
per_model.round(4)

In [ ]:
gain_ref = (best[best.horizon_s == H_REF]
            .pivot_table(index='scope', columns='feature_set', values='gain_over_persistence')
            .reindex(index=ORDER['scope'], columns=list(FEATURE_SETS)))

fig, axis = plt.subplots(figsize=(12, 5))
gain_ref.plot.bar(ax=axis, width=0.8)
axis.axhline(0, color='black', linewidth=1)
axis.set_ylabel('best accuracy − persistence')
axis.set_xlabel('')
axis.set_title(f'Gain over persistence of the best model, H={H_REF}s')
axis.legend(fontsize=8)
axis.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'gain_over_persistence_H{H_REF}.png', dpi=300, bbox_inches='tight')
plt.show()
gain_ref.round(4)

## 8. Reference case in detail

The chain is the same as at the end of the ML notebook: temporal
cross-validation on the training block, a classification report on the test
block, and the confusion matrices of the seven models side by side. The case is
set by `REF_SCOPE` and `REF_FEATURE_SET` (D4 with every store feature by
default, the scenario where the best route actually changes).

In [ ]:
ref_wide = scope_frame(REF_FEATURE_SET, REF_SCOPE)
ref = evaluate_predictive(ref_wide, H_REF, keep_models=True, quiet=False)
ref_dir = RESULTS_DIR / f'{slug(REF_SCOPE)}__{slug(REF_FEATURE_SET)}__H{H_REF}'
ref_dir.mkdir(parents=True, exist_ok=True)

print(f'{REF_SCOPE} | {REF_FEATURE_SET} | H={H_REF}s | '
      f'train={ref["n_train"]} test={ref["n_test"]} | X: {len(ref["features"])} columns')
print(f'persistence={ref["persistence"]:.4f} | majority (route {ref["majority_class"]})={ref["majority"]:.4f}')

cv_ref = temporal_cross_validation(ref['X_train'], ref['y_train'], make_pipelines())
comparison_ref = cv_ref.join(ref['results'].set_index('model').add_prefix('test_'))
comparison_ref.to_csv(ref_dir / 'cv_vs_test.csv')
comparison_ref.round(4)

In [ ]:
lines = [f'{REF_SCOPE} | {REF_FEATURE_SET} | H={H_REF}s',
         f'persistence={ref["persistence"]:.4f} majority={ref["majority"]:.4f}', '']
labels = ref['classes']
for name, model in ref['trained'].items():
    report = classification_report(ref['y_test'], model.predict(ref['X_test']), labels=labels,
                                   target_names=[f'route {label}' for label in labels], zero_division=0)
    lines += [f'--- {name}', report]
(ref_dir / 'summary.txt').write_text('\n'.join(lines), encoding='utf-8')
print('\n'.join(lines))

In [ ]:
ref_matrices = plot_confusion_grid(
    ref['trained'], ref['X_test'], ref['y_test'], ref['classes'],
    title=f'{REF_SCOPE} | {REF_FEATURE_SET} | H={H_REF}s (test block)',
    save_path=ref_dir / 'confusion_matrices.png',
)

## 9. Feature importance in the predictive regime

This is a `RandomForest` (300 trees) trained under the same protocol, with
permutation importance measured on the **test** block at `H_REF`. Each feature
is permuted on its own, and the drops in accuracy are then summed over its
routes. As a result a metric is ranked as a whole, for example
`gargalo_min_curta_15s` over the four routes. The standard deviation is taken
across the `IMPORTANCE_REPEATS` repeats of that sum.

Negative values are expected and usually small. They mean the feature carries
no signal on this test block, not that it hurts. A large negative value means
the relation learned on the training block no longer holds in the test period.

In [ ]:
def grouped_permutation_importance(wide, horizon_s=H_REF, repeats=IMPORTANCE_REPEATS):
    blocks = split_blocks(wide, horizon_s)
    model = Pipeline([('scaler', StandardScaler()),
                      ('classifier', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE,
                                                            n_jobs=-1))])
    model.fit(blocks['X_train'], blocks['y_train'])
    result = permutation_importance(model, blocks['X_test'], blocks['y_test'], n_repeats=repeats,
                                    random_state=RANDOM_STATE, n_jobs=1)
    by_metric = pd.DataFrame(result.importances, index=blocks['features']).groupby(metric_of).sum()
    return pd.DataFrame({'mean': by_metric.mean(axis=1), 'std': by_metric.std(axis=1)}), \
        accuracy_score(blocks['y_test'], model.predict(blocks['X_test']))


importances = {}
for feature_set in IMPORTANCE_FEATURE_SETS:
    for scope in SCOPES:
        table, accuracy = grouped_permutation_importance(scope_frame(feature_set, scope))
        importances[(feature_set, scope)] = table
        print(f'{feature_set:28s} | {scope:28s} | RF accuracy {accuracy:.4f}')

importance_long = pd.concat(importances, names=['feature_set', 'scope', 'metric']).reset_index()
importance_long.to_csv(RESULTS_DIR / f'permutation_importance_H{H_REF}.csv', index=False)

In [ ]:
for feature_set in IMPORTANCE_FEATURE_SETS:
    rows = importance_long[importance_long.feature_set == feature_set]
    means = rows.pivot_table(index='metric', columns='scope', values='mean').reindex(columns=list(SCOPES))
    stds = rows.pivot_table(index='metric', columns='scope', values='std').reindex(columns=list(SCOPES))
    order = means.mean(axis=1).sort_values().index
    means, stds = means.loc[order], stds.loc[order]

    fig, axis = plt.subplots(figsize=(11, max(4, 0.32 * len(order) * len(SCOPES) / 2)))
    means.plot.barh(ax=axis, xerr=stds, width=0.8, capsize=2)
    axis.axvline(0, color='black', linewidth=1)
    axis.set_xlabel('permutation importance (drop in accuracy, summed over routes)')
    axis.set_title(f'{feature_set} | H={H_REF}s | RandomForest, test block')
    axis.legend(fontsize=8)
    axis.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'permutation_importance_{slug(feature_set)}_H{H_REF}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    display(means.iloc[::-1].round(4))

## 10. Cross-dataset generalisation

This mirrors the paper's Cenário 2: the models are trained on the whole of
`CROSS_TRAIN` (D1) and tested on whole datasets of other scenarios, one group
on the original link latencies and one on the modified ones. Compared with
*Reprodução da Tabela 2* in the ML notebook, `X` now comes from the store and
has no clock columns, and the target can also be shifted (`CROSS_HORIZONS`).
At `H > 0` the target is shifted within each dataset, and persistence is
measured on each test dataset. At `H = 0` persistence is 1.0 by construction.

D1 is nearly single-class, so this setup inherits the paper's main weakness: a
model can only predict the routes it saw. `CROSS_TRAIN` can be switched to
another dataset to check how much of the result is due to that choice.

Training and test datasets must have the same routes. D3a and D4a have two, so
they can only be paired with each other (`CROSS_TRAIN = 'D3a'`,
`CROSS_TEST = {'two routes': ['D4a']}`). They are not part of the default
groups.

In [ ]:
def dataset_blocks(wide, dataset, horizon_s):
    part = wide[wide.dataset_id == dataset].dropna(axis=1, how='all').reset_index(drop=True)
    with contextlib.redirect_stdout(io.StringIO()):
        if horizon_s > 0:
            part = pt.deslocar_alvo(part, horizon_s)
        else:
            part = part.assign(**{pt.COLUNA_ALVO_ATUAL: part['melhor_rota']})
    features = pt.colunas_de_feature(part)
    return (features, part[features].to_numpy(dtype=float),
            part['melhor_rota'].to_numpy(dtype=int), part[pt.COLUNA_ALVO_ATUAL].to_numpy(dtype=int))


cross_rows = []
for feature_set, wide in wide_frames.items():
    for H in CROSS_HORIZONS:
        features, X_train, y_train, _ = dataset_blocks(wide, CROSS_TRAIN, H)
        models = {}
        for name, model in make_pipelines().items():
            try:
                models[name] = model.fit(X_train, y_train)
            except ValueError as error:
                print(f'{feature_set} H={H}: {name} not trained ({error})')

        for group, datasets in CROSS_TEST.items():
            for dataset in datasets:
                test_features, X_test, y_test, current = dataset_blocks(wide, dataset, H)
                assert test_features == features, f'{dataset}: columns differ from {CROSS_TRAIN}'
                keys = {'feature_set': feature_set, 'horizon_s': H, 'group': group, 'test': dataset}
                for name, model in models.items():
                    cross_rows.append({**keys, 'model': name,
                                       'accuracy': accuracy_score(y_test, model.predict(X_test))})
                cross_rows.append({**keys, 'model': 'baseline: persistence',
                                   'accuracy': float((current == y_test).mean())})
    print(f'{feature_set}: done')

cross = pd.DataFrame(cross_rows)
cross.to_csv(RESULTS_DIR / f'cross_dataset_train_{CROSS_TRAIN}.csv', index=False)

In [ ]:
# Mean accuracy over the test datasets of each group, as in the paper's Table 2
cross_summary = (cross.groupby(['horizon_s', 'group', 'model', 'feature_set'])['accuracy'].mean()
                 .unstack('feature_set').reindex(columns=list(FEATURE_SETS)))
row_order = MODEL_NAMES + ['baseline: persistence']
cross_summary = cross_summary.reindex(pd.MultiIndex.from_product(
    [CROSS_HORIZONS, list(CROSS_TEST), row_order], names=['horizon_s', 'group', 'model']))
print(f'Trained on {CROSS_TRAIN}, mean accuracy over each test group:')
cross_summary.round(4)

## 11. Sliding training window W

Every section so far trains once, on the whole training block, and uses that
model for the whole test block. This section asks how much history the model
actually needs. **For each evaluated test instant `t`, the models are retrained
from scratch on the last `W` seconds of labelled history only**, and then
predict `y(t + H)` from `X(t)`. `W` takes the values 5, 10, 15, 30, 45 and 60 s,
and the experiment runs for every model and every horizon `H`.

**Which samples are in the window.** A training pair `(X(τ), y(τ + H))` can only
be used at instant `t` once its label has been observed, that is, once
`τ + H ≤ t`. The window therefore ends at `τ = t − H` (at `t − 1` when `H = 0`,
so that the evaluated instant itself never enters the training set) and covers
the `W` seconds before that:

    t − H − W  <  τ  ≤  t − H

A window holds at most `W` samples, and fewer where the series has gaps. It may
reach back into the training block or into the gap that `split_temporal` leaves
before the test block. That is fine: those instants are in the past of `t`, and
an online learner would have them.

**Single-class windows.** Over a few seconds the best route rarely changes, so
many windows hold a single label. Most classifiers cannot be fitted on one class
(`LogisticRegression` and `SVC` raise an error), and any model would predict
that class anyway. Those windows predict the window's only label directly, for
every model. `single_class_%` reports how often this happens. When it is close
to 100%, every model collapses into the **window majority** baseline (the most
frequent label of the window). That baseline is reported next to persistence,
because it is exactly what a sliding-window learner gives without using `X`.

**What it is compared with.** The `full block` column is the accuracy of the same
model trained once on the whole training block, taken from the sweep of
section 6 for the same feature set and horizon. It is measured on every test
instant, while this section evaluates one instant every `WINDOW_STEP_S`
seconds, so the two match closely but not exactly.

**Cost.** Every evaluated instant fits every model again, which is why only one
instant every `WINDOW_STEP_S` seconds of the test block is evaluated and the
fits run in parallel (`WINDOW_N_JOBS`). With the defaults this is about 50
instants per scope × 6 windows × 6 horizons × 6 scopes, each with up to 7 fits
on at most 60 samples.

In [ ]:
from joblib import Parallel, delayed

WINDOW_SIZES_S = [5, 10, 15, 30, 45, 60]
WINDOW_HORIZONS = list(HORIZONS)
WINDOW_FEATURE_SET = 'store: all features'
WINDOW_SCOPES = list(SCOPES)
# One evaluated instant every WINDOW_STEP_S seconds of each dataset's test block
WINDOW_STEP_S = 30
WINDOW_N_JOBS = -1

WINDOW_DIR = RESULTS_DIR / 'training_window'
WINDOW_DIR.mkdir(parents=True, exist_ok=True)


def evaluate_window_instant(X_window, y_window, x_now, model_names=MODEL_NAMES):
    # Predictions of every model retrained on one window, plus the window majority
    labels, counts = np.unique(y_window, return_counts=True)
    majority = int(labels[counts.argmax()])  # a tie goes to the lowest route number
    out = {'window_majority': majority, 'single_class': len(labels) == 1, 'fallback': 0}

    for name, model in make_pipelines(model_names, n_jobs=1, n_samples=len(y_window)).items():
        if len(labels) == 1:
            out[name] = majority
            continue
        try:
            out[name] = int(model.fit(X_window, y_window).predict(x_now[None, :])[0])
        except ValueError:
            # e.g. a class with too few samples for the model; counted, not hidden
            out[name] = majority
            out['fallback'] += 1
    return out


def run_training_window(wide, horizon_s, window_sizes=WINDOW_SIZES_S, step_s=WINDOW_STEP_S,
                        model_names=MODEL_NAMES, n_jobs=WINDOW_N_JOBS):
    # Sorted first, so that the positions of each dataset follow time; the masks of
    #   preparar_treino_teste are aligned with the frame it returns
    ordered = wide.sort_values(['dataset_id', 'ts_epoch']).reset_index(drop=True)
    with contextlib.redirect_stdout(io.StringIO()):
        frame, _, test = pt.preparar_treino_teste(ordered, horizon_s, fracao_treino=TRAIN_FRACTION)

    features = pt.colunas_de_feature(frame)
    X = frame[features].to_numpy(dtype=float)
    y = frame['melhor_rota'].to_numpy(dtype=int)
    current = frame[pt.COLUNA_ALVO_ATUAL].to_numpy(dtype=int)
    ts = frame['ts_epoch'].to_numpy()
    test = test.to_numpy()
    last_known = max(horizon_s, 1)

    rows = []
    for dataset, positions in frame.groupby('dataset_id').indices.items():
        positions = positions[np.argsort(ts[positions])]
        ts_d = ts[positions]
        evaluated = positions[test[positions]][::step_s]

        for W in window_sizes:
            windows = []
            for i in evaluated:
                upper = ts[i] - last_known
                start = np.searchsorted(ts_d, upper - W, side='right')
                stop = np.searchsorted(ts_d, upper, side='right')
                windows.append((i, positions[start:stop]))
            windows = [(i, w) for i, w in windows if len(w)]

            outputs = Parallel(n_jobs=n_jobs)(
                delayed(evaluate_window_instant)(X[w], y[w], X[i], model_names) for i, w in windows
            )
            for (i, w), out in zip(windows, outputs):
                rows.append({'dataset': dataset, 'ts_epoch': int(ts[i]), 'window_s': W,
                             'n_train': len(w), 'y_true': int(y[i]),
                             'persistence': int(current[i]), **out})
    return pd.DataFrame(rows)


print(f'{len(WINDOW_SCOPES)} scopes x {len(WINDOW_HORIZONS)} horizons x {len(WINDOW_SIZES_S)} windows, '
      f'feature set: {WINDOW_FEATURE_SET}, one instant every {WINDOW_STEP_S} s')

In [ ]:
window_parts = []
for scope in WINDOW_SCOPES:
    wide = scope_frame(WINDOW_FEATURE_SET, scope)
    for H in WINDOW_HORIZONS:
        part = run_training_window(wide, H)
        part.insert(0, 'horizon_s', H)
        part.insert(0, 'scope', scope)
        window_parts.append(part)
        print(f'{scope:28s} | H={H:3d}s | {part.groupby("window_s").size().iloc[0]} instants per window')

window_predictions = pd.concat(window_parts, ignore_index=True)
window_predictions.to_csv(WINDOW_DIR / 'predictions.csv', index=False)
print(f'saved {len(window_predictions)} predictions to {WINDOW_DIR / "predictions.csv"}')

In [ ]:
# Accuracy and F1-macro per (scope, H, W, model); baselines and window statistics
CASE = ['scope', 'horizon_s', 'window_s']
predictors = MODEL_NAMES + ['persistence', 'window_majority']

metric_rows = []
for keys, group in window_predictions.groupby(CASE):
    for name in predictors:
        metric_rows.append({**dict(zip(CASE, keys)), 'model': name,
                            'accuracy': (group[name] == group.y_true).mean(),
                            'f1_macro': f1_score(group.y_true, group[name], average='macro', zero_division=0)})
window_metrics = pd.DataFrame(metric_rows)
window_metrics.to_csv(WINDOW_DIR / 'metrics.csv', index=False)

window_stats = window_predictions.groupby(CASE).agg(
    instants=('y_true', 'size'),
    mean_train_size=('n_train', 'mean'),
    single_class_pct=('single_class', lambda s: 100 * s.mean()),
    fallback_fits=('fallback', 'sum'),
).round(2)
window_stats.to_csv(WINDOW_DIR / 'window_stats.csv')

# Same models trained once on the whole training block (section 6), for reference
full_block = (sweep[sweep.feature_set == WINDOW_FEATURE_SET]
              .set_index(['scope', 'horizon_s', 'model'])['accuracy'])

print('Share of single-class windows (%), by window, averaged over horizons:')
window_stats.groupby(['scope', 'window_s']).single_class_pct.mean().unstack('window_s').reindex(WINDOW_SCOPES).round(1)

In [ ]:
def window_table(scope, horizon_s):
    # Accuracy per model (rows) and window (columns), with the full-block reference
    rows = window_metrics[(window_metrics.scope == scope) & (window_metrics.horizon_s == horizon_s)]
    table = rows.pivot_table(index='model', columns='window_s', values='accuracy').reindex(predictors)
    table['full block'] = [full_block.get((scope, horizon_s, m), np.nan) for m in table.index]
    return table


print(f'Accuracy per model and training window, H={H_REF}s '
      f'(baselines in the last two rows; "full block" = section 6):')
pd.concat({scope: window_table(scope, H_REF) for scope in WINDOW_SCOPES}, names=['scope', 'model']).round(4)

In [ ]:
# Best model of each case, against persistence and against the full-block model
model_rows = window_metrics[window_metrics.model.isin(MODEL_NAMES)]
best_window = model_rows.loc[model_rows.groupby(CASE)['accuracy'].idxmax()].set_index(CASE)
reference = window_metrics.pivot_table(index=CASE, columns='model', values='accuracy')

window_summary = pd.DataFrame({
    'best_model': best_window['model'],
    'best_accuracy': best_window['accuracy'],
    'persistence': reference['persistence'],
    'window_majority': reference['window_majority'],
})
window_summary['gain_over_persistence'] = window_summary.best_accuracy - window_summary.persistence
best_full = (sweep[sweep.feature_set == WINDOW_FEATURE_SET]
             .groupby(['scope', 'horizon_s']).accuracy.max())
window_summary['full_block_best'] = [best_full.get((s, h), np.nan) for s, h, _ in window_summary.index]
window_summary = window_summary.join(window_stats[['mean_train_size', 'single_class_pct']])
window_summary.to_csv(WINDOW_DIR / 'summary.csv')

print('Gain of the best window-trained model over persistence, by horizon (rows) and window (columns):')
(window_summary.gain_over_persistence.unstack('window_s')
 .reindex(pd.MultiIndex.from_product([WINDOW_SCOPES, WINDOW_HORIZONS], names=['scope', 'horizon_s']))
 .round(4))

In [ ]:
fig, axes = plt.subplots(int(np.ceil(len(WINDOW_SCOPES) / 2)), 2,
                         figsize=(14, 4.4 * np.ceil(len(WINDOW_SCOPES) / 2)), squeeze=False, sharey=True)
colours = plt.cm.viridis(np.linspace(0, 0.9, len(WINDOW_HORIZONS)))

for axis, scope in zip(axes.ravel(), WINDOW_SCOPES):
    for colour, H in zip(colours, WINDOW_HORIZONS):
        rows = window_summary.loc[(scope, H)]
        axis.plot(rows.index, rows.best_accuracy - rows.persistence, marker='o', color=colour, label=f'H={H}s')
        full_gain = best_full.get((scope, H), np.nan) - rows.persistence.mean()
        axis.axhline(full_gain, color=colour, linestyle=':', linewidth=1)
    axis.axhline(0, color='black', linewidth=1)
    axis.set_title(scope, fontsize=11)
    axis.set_xlabel('training window W (s)')
    axis.set_ylabel('best accuracy − persistence')
    axis.set_xticks(WINDOW_SIZES_S)
    axis.grid(alpha=0.3)

for axis in axes.ravel()[len(WINDOW_SCOPES):]:
    axis.axis('off')
axes.ravel()[0].legend(fontsize=8, ncol=2)
fig.suptitle(f'Sliding training window: best model vs. persistence ({WINDOW_FEATURE_SET}); '
             f'dotted = model trained on the full block', fontsize=11)
plt.tight_layout()
plt.savefig(WINDOW_DIR / 'gain_over_persistence.png', dpi=300, bbox_inches='tight')
plt.show()

## 12. Notes and limitations

- **The `H = 0` columns are not forecasts.** The target is the `argmin` of the
  instant latency, so a latency feature set at `H = 0` is a near-identity
  mapping. Only `H > 0`, read against persistence, measures predictive
  ability.
- **The store target is not the label of the label files.** `melhor_rota` comes
  from `derivar_alvo`, without the causal tie-break of
  `rotulos_menor_latencia_historico.txt` and without the bandwidth target
  (`rotulos_maior_banda.txt`). The numbers here are therefore comparable with
  the ML notebook's predictive section only for the latency target, and only
  up to ties.
- **Bandwidth unit.** The store computes bandwidth from the raw `banda.bwm` in
  decimal Mbps, while the `banda_rotas_h1_h6.csv` files of the ML notebook come
  from the Mibit/s parser. The two `bandwidth (instant)` inputs differ slightly
  in scale (see section 11 of `Feature_Store_Datasets.ipynb`).
- **D3a and D4a are not independent evidence.** They are routes 1 and 2 of D3
  and D4, built from the same raw capture (section 2.1). Their scopes show how
  the task behaves with two candidate routes instead of four. They do not
  replicate D3 and D4, and their numbers should not be averaged with those of
  D3 and D4.
- **One collection per scenario.** Each scope pools two collections (original
  and modified latencies) of the same traffic pattern. There are no repetitions
  of either, so the tables show point estimates without confidence intervals.
- **Permutation importance** uses `IMPORTANCE_REPEATS` repeats. Metrics whose
  bars overlap within one standard deviation cannot be ranked against each
  other.
- **Sliding window results are about the learner, not the features.** With
  `W` of a few seconds, most windows hold a single label and every model
  returns the window majority (section 11 reports how often). A gain at small
  `W` is therefore a property of the recent labels rather than of `X`.
- **Runtime.** Sections 6, 9 and 11 are the expensive ones. `MODEL_NAMES`,
  `HORIZONS` and `INCLUDE_POOLED_SCOPE` are the switches that shorten or extend
  them.

Outputs in `resultados_am/CenarioFeatureStore/`:

| file | content |
|---|---|
| `class_distribution_H{H}.csv` | route shares per scope and block |
| `sweep_accuracy.csv`, `sweep_baselines.csv`, `sweep_best_model.csv`, `sweep_best_model.png` | horizon sweep |
| `accuracy_per_model_H{H}.csv`, `gain_over_persistence_H{H}.png` | feature-set comparison |
| `<scope>__<set>__H{H}/` | reference case: `cv_vs_test.csv`, `summary.txt`, `confusion_matrices.png` |
| `permutation_importance_*` | feature importance |
| `cross_dataset_train_{D}.csv` | cross-dataset generalisation |
| `training_window/` | sliding training window: `predictions.csv`, `metrics.csv`, `window_stats.csv`, `summary.csv`, `gain_over_persistence.png` |